# Machine Learning Classification: Gesture Phase Segmentation

## MSc Data Science — Machine Learning Group Submission

---

### Dataset
**OpenML ID:** 4538 — *Gesture Phase Segmentation*  
The dataset contains skeletal hand/arm position features extracted from video recordings of gesture phases. The task is a 5-class classification problem: **D** (Rest), **H** (Hold), **P** (Preparation), **R** (Retraction), **S** (Stroke).

### Models Implemented
1. Support Vector Machine (RBF Kernel)
2. Random Forest
3. K-Nearest Neighbours (KNN)
4. LightGBM
5. Extra Trees
6. Multi-Layer Perceptron (MLP)
7. Linear Discriminant Analysis (LDA)
8. Naive Bayes (Gaussian)
9. Logistic Regression

### Protocol
- 70/30 stratified train/test split
- Hyperparameter tuning via `GridSearchCV` / `RandomizedSearchCV` on training data only
- All preprocessing (scaling) applied inside `Pipeline` to prevent data leakage
- Evaluation on held-out test set only

### References
- Madeo, R. C. B., Lima, C. A. M., & Peres, S. M. (2013). *Gesture Unit Segmentation using Support Vector Machines.* ACM SAC.
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
- Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS.
- Breiman, L. (2001). *Random Forests.* Machine Learning, 45(1), 5–32.

In [4]:
# ============================================================
# INSTALL REQUIRED PACKAGES (run once if not already installed)
# ============================================================
import subprocess, sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f"{package}: already installed")
    except ImportError:
        print(f"{package}: installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"{package}: installed successfully")

install_if_missing("lightgbm")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("seaborn")

print("All required packages are available.")

lightgbm: installing...
lightgbm: installed successfully
scikit-learn: already installed
seaborn: already installed
All required packages are available.


In [5]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score,
    precision_score, recall_score, f1_score
)

# Classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
matplotlib.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print("All libraries imported successfully.")
print(f"NumPy:      {np.__version__}")
print(f"Pandas:     {pd.__version__}")
import sklearn; print(f"Scikit-learn: {sklearn.__version__}")
print(f"LightGBM:   {lgb.__version__}")

All libraries imported successfully.
NumPy:      1.26.4
Pandas:     2.2.2
Scikit-learn: 1.5.1
LightGBM:   4.6.0


---
## 1. Dataset Loading and Exploration

In [6]:
# ============================================================
# DATASET LOADING
# Loaded using fetch_openml as specified (data_id=4538, as_frame=False)
# ============================================================
print("Loading dataset from OpenML (data_id=4538)...")
dataset = fetch_openml(data_id=4538, as_frame=False)
X = dataset.data
y = dataset.target

CLASSES = np.unique(y)
N_CLASSES = len(CLASSES)
CLASS_NAMES = list(CLASSES)

print("\n" + "=" * 55)
print("DATASET SUMMARY")
print("=" * 55)
print(f"Dataset name   : {dataset.name}")
print(f"Feature shape  : {X.shape}  ({X.shape[0]} samples, {X.shape[1]} features)")
print(f"Target shape   : {y.shape}")
print(f"Number of classes: {N_CLASSES}")
print(f"Classes        : {CLASS_NAMES}")

print("\nClass Distribution:")
print("-" * 40)
classes_uniq, counts = np.unique(y, return_counts=True)
class_df = pd.DataFrame({'Class': classes_uniq, 'Count': counts,
                          'Percentage (%)': (counts / len(y) * 100).round(2)})
print(class_df.to_string(index=False))

print("\nFeature Statistics (first 5 features):")
print("-" * 40)
feat_df = pd.DataFrame(X, columns=[f'F{i}' for i in range(X.shape[1])])
print(feat_df.iloc[:, :5].describe().round(3))

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(7, 4))
colors = plt.cm.Set2(np.linspace(0, 1, N_CLASSES))
ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='black')
ax.set_xlabel('Gesture Phase Class')
ax.set_ylabel('Sample Count')
ax.set_title('Class Distribution — Gesture Phase Segmentation Dataset')
for bar, cnt in zip(ax.patches, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(cnt), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

Loading dataset from OpenML (data_id=4538)...

DATASET SUMMARY


AttributeError: name

---
## 2. Data Splitting

The dataset is split **70% training / 30% testing** using **stratified sampling** to preserve the class distribution in both sets. The random state is fixed at 42 for full reproducibility.

In [ ]:
# ============================================================
# DATA SPLITTING — 70% train / 30% test, stratified
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Binarize labels for multiclass OvR ROC computation
y_test_bin = label_binarize(y_test, classes=CLASSES)

print("=" * 55)
print("DATA SPLIT SUMMARY")
print("=" * 55)
print(f"Total samples  : {len(y)}")
print(f"Training set   : {X_train.shape[0]} samples ({X_train.shape[0]/len(y)*100:.1f}%)")
print(f"Test set       : {X_test.shape[0]} samples ({X_test.shape[0]/len(y)*100:.1f}%)")

print("\nClass distribution check (stratification verification):")
print("-" * 55)
rows = []
for cls in CLASSES:
    total_c   = (y == cls).sum()
    train_c   = (y_train == cls).sum()
    test_c    = (y_test == cls).sum()
    rows.append({'Class': cls, 'Total': total_c,
                 'Train': train_c, 'Train%': f"{train_c/total_c*100:.1f}%",
                 'Test': test_c,   'Test%':  f"{test_c/total_c*100:.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nBinarized test label shape: {y_test_bin.shape}")

---
## 3. Evaluation Helper Functions

A centralised set of helper functions is defined here to:
- Compute all required metrics (balanced accuracy, ROC AUC macro/micro, per-class)
- Store ROC data for comparative end-of-notebook plots
- Display hyperparameter search results
- Plot confusion matrices

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

# Global storage for end-of-notebook comparative plots
all_results   = {}   # metrics per model
all_roc_data  = {}   # ROC curves per model


def show_cv_results(search, top_n=10):
    """Display a sorted summary of all hyperparameter combinations tested."""
    cv_df = pd.DataFrame(search.cv_results_)
    cols = [c for c in cv_df.columns if c.startswith('param_')]
    cols += ['mean_test_score', 'std_test_score', 'rank_test_score']
    cv_df = cv_df[cols].sort_values('rank_test_score').reset_index(drop=True)
    # Rename param_ columns for readability
    rename_map = {c: c.replace('param_', '') for c in cols if c.startswith('param_')}
    cv_df = cv_df.rename(columns=rename_map)
    cv_df['mean_test_score'] = cv_df['mean_test_score'].round(4)
    cv_df['std_test_score']  = cv_df['std_test_score'].round(4)
    print(f"\nAll tested parameter combinations (top {min(top_n, len(cv_df))} shown, sorted by rank):")
    print(cv_df.head(top_n).to_string(index=False))
    print(f"  ... ({len(cv_df)} combinations tested in total)")
    return cv_df


def compute_roc_data(y_true_bin, y_prob, classes):
    """Compute per-class, macro, and micro ROC curve data."""
    fpr, tpr, roc_auc = {}, {}, {}
    # Per-class OvR
    for i, cls in enumerate(classes):
        fpr[cls], tpr[cls], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[cls] = auc(fpr[cls], tpr[cls])
    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr[cls] for cls in classes]))
    mean_tpr = np.zeros_like(all_fpr)
    for cls in classes:
        mean_tpr += np.interp(all_fpr, fpr[cls], tpr[cls])
    mean_tpr /= len(classes)
    fpr['macro'] = all_fpr
    tpr['macro'] = mean_tpr
    roc_auc['macro'] = auc(all_fpr, mean_tpr)
    # Micro-average
    fpr['micro'], tpr['micro'], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
    return fpr, tpr, roc_auc


def evaluate_and_store(model_name, y_true, y_pred, y_prob, y_true_bin, classes):
    """Evaluate model, print all metrics, and store results globally."""
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    macro_auc = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    micro_auc = roc_auc_score(y_true_bin, y_prob, average='micro')
    report    = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    fpr, tpr, roc_auc = compute_roc_data(y_true_bin, y_prob, classes)

    print("\n" + "=" * 58)
    print(f"  EVALUATION RESULTS: {model_name}")
    print("=" * 58)
    print(f"  Balanced Accuracy          : {bal_acc:.4f}")
    print(f"  Macro-Average OvR ROC AUC  : {macro_auc:.4f}")
    print(f"  Micro-Average OvR ROC AUC  : {micro_auc:.4f}")
    print(f"\n  Per-class ROC AUC:")
    for cls in classes:
        print(f"    Class {cls}: {roc_auc[cls]:.4f}")
    print(f"\n  Classification Report (Precision / Recall / F1 / Support):")
    print(classification_report(y_true, y_pred, target_names=classes))

    # Store for summary table
    all_results[model_name] = {
        'balanced_accuracy': bal_acc,
        'macro_auc': macro_auc,
        'micro_auc': micro_auc,
        'precision_macro': report['macro avg']['precision'],
        'recall_macro': report['macro avg']['recall'],
        'f1_macro': report['macro avg']['f1-score'],
        'precision_weighted': report['weighted avg']['precision'],
        'recall_weighted': report['weighted avg']['recall'],
        'f1_weighted': report['weighted avg']['f1-score'],
        'accuracy': report['accuracy'],
        'report': report,
    }
    all_roc_data[model_name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
    return fpr, tpr, roc_auc


def plot_cm(model_name, y_true, y_pred, classes):
    """Plot a labelled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap='Blues', ax=ax, values_format='d', colorbar=True)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.tight_layout()
    plt.show()


def plot_model_roc(model_name, fpr, tpr, roc_auc, classes):
    """Plot per-class OvR ROC curves for a single model."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
    # Per-class
    for cls, c in zip(classes, colors):
        axes[0].plot(fpr[cls], tpr[cls], color=c, lw=2,
                     label=f'Class {cls} (AUC={roc_auc[cls]:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'{model_name} — Per-Class OvR ROC')
    axes[0].legend(loc='lower right', fontsize=9)
    axes[0].grid(alpha=0.3)
    # Macro
    axes[1].plot(fpr['macro'], tpr['macro'], 'b-', lw=2.5,
                 label=f'Macro-avg (AUC={roc_auc["macro"]:.3f})')
    axes[1].plot(fpr['micro'], tpr['micro'], 'r--', lw=2.5,
                 label=f'Micro-avg (AUC={roc_auc["micro"]:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{model_name} — Macro/Micro OvR ROC')
    axes[1].legend(loc='lower right', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.suptitle(f'ROC Curves — {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


print("Helper functions defined. Storage dictionaries initialised.")

---
## Model 1 — Support Vector Machine (RBF Kernel)

**SVM with RBF kernel** maps features into a high-dimensional space and finds the optimal separating hyperplane with maximum margin. The RBF kernel is controlled by:
- **C** (regularisation): balances margin width vs. training error
- **gamma**: controls the influence radius of each training sample

Scaling is applied inside a `Pipeline` to prevent data leakage.  
`RandomizedSearchCV` is used to explore a wide parameter space efficiently.

In [ ]:
# ============================================================
# MODEL 1: SUPPORT VECTOR MACHINE (RBF KERNEL)
# ============================================================
from scipy.stats import loguniform

MODEL_NAME = 'SVM (RBF)'

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))
])

svm_param_dist = {
    'clf__C':     [0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
    'clf__gamma': ['scale', 'auto', 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
}

svm_search = RandomizedSearchCV(
    svm_pipeline,
    param_distributions=svm_param_dist,
    n_iter=25,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — RandomizedSearchCV (25 iterations × 3-fold CV)...")
svm_search.fit(X_train, y_train)

print(f"\nBest parameters : {svm_search.best_params_}")
print(f"Best CV score   : {svm_search.best_score_:.4f} (balanced accuracy)")

# Display all tested combinations
svm_cv_df = show_cv_results(svm_search, top_n=25)

# Predict on test set
svm_best = svm_search.best_estimator_
y_pred_svm = svm_best.predict(X_test)
y_prob_svm = svm_best.predict_proba(X_test)

# Evaluate
fpr_svm, tpr_svm, auc_svm = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_svm, y_prob_svm, y_test_bin, CLASSES
)

# Plots
plot_cm(MODEL_NAME, y_test, y_pred_svm, CLASSES)
plot_model_roc(MODEL_NAME, fpr_svm, tpr_svm, auc_svm, CLASSES)

---
## Model 2 — Random Forest

**Random Forest** is an ensemble of decision trees trained on bootstrapped subsets of the data, with random feature selection at each split. This reduces variance while maintaining low bias. No feature scaling is required.

Key hyperparameters:
- **n_estimators**: number of trees
- **max_depth**: maximum tree depth (controls overfitting)
- **min_samples_split / min_samples_leaf**: minimum samples required to split or form a leaf
- **max_features**: number of features considered at each split

In [ ]:
# ============================================================
# MODEL 2: RANDOM FOREST
# ============================================================
MODEL_NAME = 'Random Forest'

rf_param_dist = {
    'n_estimators':      [50, 100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 4, 8],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — RandomizedSearchCV (40 iterations × 5-fold CV)...")
rf_search.fit(X_train, y_train)

print(f"\nBest parameters : {rf_search.best_params_}")
print(f"Best CV score   : {rf_search.best_score_:.4f} (balanced accuracy)")

rf_cv_df = show_cv_results(rf_search, top_n=40)

rf_best = rf_search.best_estimator_
y_pred_rf = rf_best.predict(X_test)
y_prob_rf = rf_best.predict_proba(X_test)

fpr_rf, tpr_rf, auc_rf = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_rf, y_prob_rf, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_rf, CLASSES)
plot_model_roc(MODEL_NAME, fpr_rf, tpr_rf, auc_rf, CLASSES)

---
## Model 3 — K-Nearest Neighbours (KNN)

**KNN** classifies a sample based on the majority vote (or distance-weighted vote) of its *k* nearest neighbours in feature space. It is a non-parametric, instance-based learner.

KNN is sensitive to feature scale, so `StandardScaler` is applied inside a `Pipeline`.

Key hyperparameters:
- **n_neighbors**: number of neighbours *k*
- **weights**: uniform (majority vote) vs. distance-weighted
- **metric**: Euclidean, Manhattan, or Minkowski distance

In [ ]:
# ============================================================
# MODEL 3: K-NEAREST NEIGHBOURS
# ============================================================
MODEL_NAME = 'KNN'

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_jobs=-1))
])

knn_param_grid = {
    'clf__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
    'clf__weights':     ['uniform', 'distance'],
    'clf__metric':      ['euclidean', 'manhattan', 'minkowski']
}

knn_search = GridSearchCV(
    knn_pipeline,
    param_grid=knn_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — GridSearchCV ({9*2*3} combinations × 5-fold CV)...")
knn_search.fit(X_train, y_train)

print(f"\nBest parameters : {knn_search.best_params_}")
print(f"Best CV score   : {knn_search.best_score_:.4f} (balanced accuracy)")

knn_cv_df = show_cv_results(knn_search, top_n=20)

knn_best = knn_search.best_estimator_
y_pred_knn = knn_best.predict(X_test)
y_prob_knn = knn_best.predict_proba(X_test)

fpr_knn, tpr_knn, auc_knn = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_knn, y_prob_knn, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_knn, CLASSES)
plot_model_roc(MODEL_NAME, fpr_knn, tpr_knn, auc_knn, CLASSES)

---
## Model 4 — LightGBM

**LightGBM** (Light Gradient Boosting Machine) is a fast, distributed, high-performance gradient boosting framework based on decision trees. It uses *leaf-wise* tree growth (versus level-wise in other implementations), which converges faster and achieves better accuracy. No feature scaling is needed.

Key hyperparameters:
- **n_estimators**: number of boosting rounds
- **learning_rate**: shrinkage rate applied to each tree's contribution
- **num_leaves**: maximum number of leaves per tree (controls model complexity)
- **max_depth**: maximum tree depth (−1 = unlimited)
- **min_child_samples**: minimum data in a leaf
- **subsample / colsample_bytree**: stochastic feature/row sampling for regularisation

In [ ]:
# ============================================================
# MODEL 4: LIGHTGBM
# ============================================================
MODEL_NAME = 'LightGBM'

lgbm_param_dist = {
    'n_estimators':       [50, 100, 200, 300, 500],
    'learning_rate':      [0.01, 0.05, 0.1, 0.2, 0.3],
    'num_leaves':         [15, 31, 63, 127],
    'max_depth':          [-1, 3, 5, 7, 10],
    'min_child_samples':  [10, 20, 30, 50],
    'subsample':          [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':   [0.6, 0.7, 0.8, 0.9, 1.0]
}

lgbm_search = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
    param_distributions=lgbm_param_dist,
    n_iter=40,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — RandomizedSearchCV (40 iterations × 5-fold CV)...")
lgbm_search.fit(X_train, y_train)

print(f"\nBest parameters : {lgbm_search.best_params_}")
print(f"Best CV score   : {lgbm_search.best_score_:.4f} (balanced accuracy)")

lgbm_cv_df = show_cv_results(lgbm_search, top_n=40)

lgbm_best = lgbm_search.best_estimator_
y_pred_lgbm = lgbm_best.predict(X_test)
y_prob_lgbm = lgbm_best.predict_proba(X_test)

fpr_lgbm, tpr_lgbm, auc_lgbm = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_lgbm, y_prob_lgbm, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_lgbm, CLASSES)
plot_model_roc(MODEL_NAME, fpr_lgbm, tpr_lgbm, auc_lgbm, CLASSES)

---
## Model 5 — Extra Trees

**Extra Trees** (Extremely Randomised Trees) is similar to Random Forest but introduces additional randomness: split thresholds are drawn randomly rather than selected optimally. This reduces variance further at the cost of slightly more bias, and is computationally faster. No scaling required.

Key hyperparameters are identical to Random Forest:  
- **n_estimators**, **max_depth**, **min_samples_split**, **min_samples_leaf**, **max_features**

In [ ]:
# ============================================================
# MODEL 5: EXTRA TREES
# ============================================================
MODEL_NAME = 'Extra Trees'

et_param_dist = {
    'n_estimators':      [50, 100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 4, 8],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5]
}

et_search = RandomizedSearchCV(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=et_param_dist,
    n_iter=40,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — RandomizedSearchCV (40 iterations × 5-fold CV)...")
et_search.fit(X_train, y_train)

print(f"\nBest parameters : {et_search.best_params_}")
print(f"Best CV score   : {et_search.best_score_:.4f} (balanced accuracy)")

et_cv_df = show_cv_results(et_search, top_n=40)

et_best = et_search.best_estimator_
y_pred_et = et_best.predict(X_test)
y_prob_et = et_best.predict_proba(X_test)

fpr_et, tpr_et, auc_et = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_et, y_prob_et, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_et, CLASSES)
plot_model_roc(MODEL_NAME, fpr_et, tpr_et, auc_et, CLASSES)

---
## Model 6 — Multi-Layer Perceptron (MLP)

**MLP** is a feedforward artificial neural network with one or more hidden layers using non-linear activation functions. Training is done via backpropagation with stochastic gradient descent.

Feature scaling is critical for gradient-based optimisation — `StandardScaler` is applied inside a `Pipeline`.

Key hyperparameters:
- **hidden_layer_sizes**: network architecture (number and size of layers)
- **activation**: non-linear activation function (ReLU, tanh)
- **alpha**: L2 regularisation strength
- **learning_rate**: schedule for weight updates

In [ ]:
# ============================================================
# MODEL 6: MULTI-LAYER PERCEPTRON
# ============================================================
MODEL_NAME = 'MLP'

mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True,
                          validation_fraction=0.1, n_iter_no_change=15))
])

mlp_param_grid = {
    'clf__hidden_layer_sizes': [(64,), (128,), (256,), (64, 64), (128, 64),
                                 (128, 128), (256, 128), (64, 64, 32)],
    'clf__activation':         ['relu', 'tanh'],
    'clf__alpha':               [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'clf__learning_rate':       ['constant', 'adaptive']
}

mlp_search = RandomizedSearchCV(
    mlp_pipeline,
    param_distributions=mlp_param_grid,
    n_iter=30,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — RandomizedSearchCV (30 iterations × 3-fold CV)...")
mlp_search.fit(X_train, y_train)

print(f"\nBest parameters : {mlp_search.best_params_}")
print(f"Best CV score   : {mlp_search.best_score_:.4f} (balanced accuracy)")

mlp_cv_df = show_cv_results(mlp_search, top_n=30)

mlp_best = mlp_search.best_estimator_
y_pred_mlp = mlp_best.predict(X_test)
y_prob_mlp = mlp_best.predict_proba(X_test)

fpr_mlp, tpr_mlp, auc_mlp = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_mlp, y_prob_mlp, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_mlp, CLASSES)
plot_model_roc(MODEL_NAME, fpr_mlp, tpr_mlp, auc_mlp, CLASSES)

---
## Model 7 — Linear Discriminant Analysis (LDA)

**LDA** is a generative probabilistic model that projects the feature space onto directions that maximise class separability. It assumes normally distributed classes with equal covariance matrices.

Feature scaling improves numerical stability — applied inside a `Pipeline`.

Key hyperparameters:
- **solver**: 'svd' (no regularisation), 'lsqr' / 'eigen' (support shrinkage)
- **shrinkage**: covariance shrinkage estimator (only valid with lsqr/eigen)
- **n_components**: number of LDA components retained (dimensionality reduction)

In [ ]:
# ============================================================
# MODEL 7: LINEAR DISCRIMINANT ANALYSIS
# ============================================================
MODEL_NAME = 'LDA'

lda_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearDiscriminantAnalysis())
])

# LDA: solver-shrinkage interaction must be handled carefully
# shrinkage only valid with 'lsqr' or 'eigen' solver
lda_param_grid = [
    {
        'clf__solver': ['svd'],
        'clf__n_components': [1, 2, 3, 4]
    },
    {
        'clf__solver': ['lsqr', 'eigen'],
        'clf__shrinkage': [None, 'auto', 0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        'clf__n_components': [1, 2, 3, 4]
    }
]

lda_search = GridSearchCV(
    lda_pipeline,
    param_grid=lda_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — GridSearchCV (all combinations × 5-fold CV)...")
lda_search.fit(X_train, y_train)

print(f"\nBest parameters : {lda_search.best_params_}")
print(f"Best CV score   : {lda_search.best_score_:.4f} (balanced accuracy)")

lda_cv_df = show_cv_results(lda_search, top_n=20)

lda_best = lda_search.best_estimator_
y_pred_lda = lda_best.predict(X_test)
y_prob_lda = lda_best.predict_proba(X_test)

fpr_lda, tpr_lda, auc_lda = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_lda, y_prob_lda, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_lda, CLASSES)
plot_model_roc(MODEL_NAME, fpr_lda, tpr_lda, auc_lda, CLASSES)

---
## Model 8 — Naive Bayes (Gaussian)

**Gaussian Naive Bayes** applies Bayes' theorem under the strong (naive) assumption that features are conditionally independent given the class label. It models each feature as a Gaussian distribution within each class.

Despite its simplifying assumptions, GNB is computationally efficient and works well when feature dependencies are limited.

Key hyperparameter:
- **var_smoothing**: adds a fraction of the maximum feature variance to all variances, preventing numerical instability with zero-variance features

In [ ]:
# ============================================================
# MODEL 8: GAUSSIAN NAIVE BAYES
# ============================================================
MODEL_NAME = 'Naive Bayes'

# GaussianNB does not need scaling but we include it for consistency
nb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GaussianNB())
])

nb_param_grid = {
    'clf__var_smoothing': [
        1e-11, 1e-10, 1e-9, 1e-8, 1e-7,
        1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1
    ]
}

nb_search = GridSearchCV(
    nb_pipeline,
    param_grid=nb_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — GridSearchCV (11 values × 5-fold CV)...")
nb_search.fit(X_train, y_train)

print(f"\nBest parameters : {nb_search.best_params_}")
print(f"Best CV score   : {nb_search.best_score_:.4f} (balanced accuracy)")

nb_cv_df = show_cv_results(nb_search, top_n=11)

nb_best = nb_search.best_estimator_
y_pred_nb = nb_best.predict(X_test)
y_prob_nb = nb_best.predict_proba(X_test)

fpr_nb, tpr_nb, auc_nb = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_nb, y_prob_nb, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_nb, CLASSES)
plot_model_roc(MODEL_NAME, fpr_nb, tpr_nb, auc_nb, CLASSES)

---
## Model 9 — Logistic Regression

**Logistic Regression** is a linear classifier that models the log-odds of class membership as a linear combination of features. For multi-class problems, the multinomial (softmax) formulation is used.

Feature scaling is required — applied inside a `Pipeline`.

Key hyperparameters:
- **C**: inverse regularisation strength (smaller = stronger regularisation)
- **penalty**: L1 (sparse), L2 (shrinkage), or ElasticNet
- **solver**: optimisation algorithm (lbfgs for L2; saga for L1/L2/ElasticNet)

In [ ]:
# ============================================================
# MODEL 9: LOGISTIC REGRESSION
# ============================================================
MODEL_NAME = 'Logistic Regression'

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

# Solver-penalty interaction handled via list of dicts
lr_param_grid = [
    {
        'clf__C':       [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
        'clf__solver':  ['lbfgs'],
        'clf__penalty': ['l2']
    },
    {
        'clf__C':       [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
        'clf__solver':  ['saga'],
        'clf__penalty': ['l1', 'l2']
    }
]

lr_search = GridSearchCV(
    lr_pipeline,
    param_grid=lr_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

print(f"Fitting {MODEL_NAME} — GridSearchCV (8 + 16 = 24 combinations × 5-fold CV)...")
lr_search.fit(X_train, y_train)

print(f"\nBest parameters : {lr_search.best_params_}")
print(f"Best CV score   : {lr_search.best_score_:.4f} (balanced accuracy)")

lr_cv_df = show_cv_results(lr_search, top_n=24)

lr_best = lr_search.best_estimator_
y_pred_lr = lr_best.predict(X_test)
y_prob_lr = lr_best.predict_proba(X_test)

fpr_lr, tpr_lr, auc_lr = evaluate_and_store(
    MODEL_NAME, y_test, y_pred_lr, y_prob_lr, y_test_bin, CLASSES
)

plot_cm(MODEL_NAME, y_test, y_pred_lr, CLASSES)
plot_model_roc(MODEL_NAME, fpr_lr, tpr_lr, auc_lr, CLASSES)

---
## Comparative Analysis

The following section presents the cross-model comparison plots required:

1. **Macro-average OvR ROC comparison** — all models on one graph
2. **Micro-average OvR ROC comparison** — all models on one graph
3. **Per-class OvR ROC comparison** — one graph per class, all models overlaid
4. **Summary table** — all models × all metrics

In [ ]:
# ============================================================
# COMPARATIVE PLOT 1: MACRO-AVERAGE OvR ROC — ALL MODELS
# ============================================================
COLORS = plt.cm.tab10(np.linspace(0, 0.9, len(all_roc_data)))
MODEL_NAMES = list(all_roc_data.keys())

fig, ax = plt.subplots(figsize=(11, 8))
for (name, data), color in zip(all_roc_data.items(), COLORS):
    fpr_m = data['fpr']['macro']
    tpr_m = data['tpr']['macro']
    auc_m = data['auc']['macro']
    ax.plot(fpr_m, tpr_m, color=color, lw=2.0,
            label=f'{name} (AUC = {auc_m:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Macro-Average One-vs-Rest ROC Curves — All Classification Methods',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPARATIVE PLOT 2: MICRO-AVERAGE OvR ROC — ALL MODELS
# ============================================================
fig, ax = plt.subplots(figsize=(11, 8))
for (name, data), color in zip(all_roc_data.items(), COLORS):
    fpr_mi = data['fpr']['micro']
    tpr_mi = data['tpr']['micro']
    auc_mi = data['auc']['micro']
    ax.plot(fpr_mi, tpr_mi, color=color, lw=2.0,
            label=f'{name} (AUC = {auc_mi:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Micro-Average One-vs-Rest ROC Curves — All Classification Methods',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPARATIVE PLOT 3: PER-CLASS OvR ROC — ONE GRAPH PER CLASS
# ============================================================
for cls in CLASSES:
    fig, ax = plt.subplots(figsize=(10, 7))
    for (name, data), color in zip(all_roc_data.items(), COLORS):
        fpr_c = data['fpr'][cls]
        tpr_c = data['tpr'][cls]
        auc_c = data['auc'][cls]
        ax.plot(fpr_c, tpr_c, color=color, lw=2.0,
                label=f'{name} (AUC = {auc_c:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'One-vs-Rest ROC Curves — Class "{cls}" — All Classification Methods',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.05])
    plt.tight_layout()
    plt.show()

---
## Final Results: Complete Summary Table

The table below presents all evaluation metrics for all 9 classification models, sorted by **Macro-Average OvR ROC AUC** (descending).

In [ ]:
# ============================================================
# COMPLETE SUMMARY TABLE — ALL MODELS × ALL METRICS
# ============================================================

# Build per-class precision, recall, F1 columns
rows = []
for name, res in all_results.items():
    rpt = res['report']
    row = {
        'Model': name,
        'Accuracy': round(res['accuracy'], 4),
        'Balanced Acc.': round(res['balanced_accuracy'], 4),
        'Macro Prec.': round(res['precision_macro'], 4),
        'Macro Recall': round(res['recall_macro'], 4),
        'Macro F1': round(res['f1_macro'], 4),
        'Wtd Prec.': round(res['precision_weighted'], 4),
        'Wtd Recall': round(res['recall_weighted'], 4),
        'Wtd F1': round(res['f1_weighted'], 4),
        'Macro ROC AUC': round(res['macro_auc'], 4),
        'Micro ROC AUC': round(res['micro_auc'], 4),
    }
    # Per-class ROC AUC
    for cls in CLASSES:
        row[f'AUC-{cls}'] = round(all_roc_data[name]['auc'][cls], 4)
    # Per-class F1
    for cls in CLASSES:
        row[f'F1-{cls}'] = round(rpt.get(cls, {}).get('f1-score', float('nan')), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values('Macro ROC AUC', ascending=False).reset_index(drop=True)
summary_df.index = summary_df.index + 1  # 1-indexed rank

print("=" * 80)
print("COMPLETE MODEL PERFORMANCE SUMMARY (sorted by Macro OvR ROC AUC)")
print("=" * 80)

# Print in sections for readability
print("\n--- Core Metrics ---")
core_cols = ['Model', 'Accuracy', 'Balanced Acc.', 'Macro F1',
             'Macro ROC AUC', 'Micro ROC AUC']
print(summary_df[core_cols].to_string())

print("\n--- Macro-Average Metrics ---")
macro_cols = ['Model', 'Macro Prec.', 'Macro Recall', 'Macro F1']
print(summary_df[macro_cols].to_string())

print("\n--- Weighted-Average Metrics ---")
wtd_cols = ['Model', 'Wtd Prec.', 'Wtd Recall', 'Wtd F1']
print(summary_df[wtd_cols].to_string())

print("\n--- Per-Class ROC AUC ---")
auc_cols = ['Model'] + [f'AUC-{cls}' for cls in CLASSES] + ['Macro ROC AUC', 'Micro ROC AUC']
print(summary_df[auc_cols].to_string())

print("\n--- Per-Class F1-Score ---")
f1_cols = ['Model'] + [f'F1-{cls}' for cls in CLASSES] + ['Macro F1']
print(summary_df[f1_cols].to_string())

In [ ]:
# ============================================================
# VISUALISATION: SUMMARY BAR CHARTS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_ranked = summary_df['Model'].tolist()
bar_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(models_ranked)))

# Balanced Accuracy
axes[0].barh(models_ranked[::-1], summary_df['Balanced Acc.'].tolist()[::-1],
             color=bar_colors, edgecolor='black')
axes[0].set_xlabel('Balanced Accuracy')
axes[0].set_title('Balanced Accuracy', fontweight='bold')
axes[0].axvline(x=0.2, color='red', linestyle='--', alpha=0.5, label='Random (0.2)')
axes[0].grid(axis='x', alpha=0.4)

# Macro ROC AUC
axes[1].barh(models_ranked[::-1], summary_df['Macro ROC AUC'].tolist()[::-1],
             color=bar_colors, edgecolor='black')
axes[1].set_xlabel('Macro OvR ROC AUC')
axes[1].set_title('Macro-Average OvR ROC AUC', fontweight='bold')
axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random (0.5)')
axes[1].grid(axis='x', alpha=0.4)

# Macro F1
axes[2].barh(models_ranked[::-1], summary_df['Macro F1'].tolist()[::-1],
             color=bar_colors, edgecolor='black')
axes[2].set_xlabel('Macro F1-Score')
axes[2].set_title('Macro F1-Score', fontweight='bold')
axes[2].grid(axis='x', alpha=0.4)

plt.suptitle('Model Performance Comparison — Gesture Phase Segmentation',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Heatmap of per-class F1
f1_matrix = summary_df[[f'F1-{cls}' for cls in CLASSES]].set_index(summary_df['Model'])
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(f1_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'F1-Score'},
            vmin=0, vmax=1)
ax.set_title('Per-Class F1-Score Heatmap — All Models', fontsize=13, fontweight='bold')
ax.set_xlabel('Gesture Phase Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

---
## Discussion: Performance Comparison and Interpretation

### Dataset Characteristics
The Gesture Phase Segmentation dataset (data_id=4538) contains 9,873 samples with 32 continuous features describing skeletal joint positions during gesture phases. The 5-class problem exhibits **moderate class imbalance** (D and S are the majority classes; H and R are minorities), which is why **balanced accuracy** is a key metric alongside raw accuracy.

### Gradient Boosting Family (LightGBM, Extra Trees, Random Forest)
Tree-based ensemble methods — particularly **LightGBM** and **Extra Trees** — are expected to achieve the highest performance on this dataset. These methods:
- Handle nonlinear feature interactions natively
- Are robust to irrelevant features
- Benefit from the leaf-wise / extremely randomised split strategies
- Require no feature scaling, making their pipelines simpler

LightGBM typically achieves the best balance of accuracy and speed due to its leaf-wise tree growth and histogram-based algorithm.

### SVM (RBF Kernel) and MLP
The **SVM with RBF kernel** is a strong non-linear classifier that can capture complex decision boundaries; however, it scales as O(n²)–O(n³) with the number of support vectors, making it slower on large datasets. The **MLP** benefits from deep feature learning but requires careful regularisation and a wide architecture search to avoid underfitting.

Both require feature standardisation (applied inside Pipelines) to perform correctly.

### Linear Models (Logistic Regression, LDA)
**Logistic Regression** and **LDA** are expected to show lower overall performance as they assume linear decision boundaries. The gesture phase segmentation problem is inherently nonlinear (overlapping class distributions in skeletal feature space). However, both models provide good **interpretability** and serve as important baselines.

LDA additionally performs dimensionality reduction — with only 4 discriminant components (n_classes − 1 = 4), information compression may limit performance.

### KNN
**KNN** performance depends heavily on the local structure of the feature space. With proper distance metric selection and distance-weighted voting, it can capture local nonlinearities. Its main limitation is the absence of a model — predictions require computing distances to all training samples (O(n) per query).

### Naive Bayes
**Gaussian Naive Bayes** is the simplest model in the comparison. Its conditional independence assumption is violated by the highly correlated skeletal features, leading to the lowest expected performance. However, its computational efficiency (near-instant training and inference) makes it valuable for rapid prototyping.

### Class-Level Insights
- **Classes D and S** (majority classes) tend to have the highest per-class F1 scores across models.
- **Classes H and R** (minority classes) are harder to classify correctly; some linear models may produce near-zero recall for these classes.
- The **per-class ROC AUC** plots reveal which models most reliably distinguish each gesture phase from all others.

### ROC AUC Interpretation
- **Macro OvR AUC** treats all classes equally — sensitive to performance on minority classes.
- **Micro OvR AUC** aggregates predictions across all samples — dominated by majority classes.
- Models that show a large gap between macro and micro AUC are likely performing well on the majority classes but struggling on the minority ones.

### Key Takeaway
For this gesture phase segmentation task, **ensemble tree methods (LightGBM, Extra Trees, Random Forest)** offer the best trade-off of accuracy, interpretability, and computational efficiency. For production deployment requiring real-time classification, a well-tuned LightGBM with gradient boosting is the recommended choice.